# Gold — Indicator Dimension (Kimball)

Describes each economic indicator measured in `gold.fact_monthly_economics`.
Includes descriptive attributes and a foreign key to `gold.dim_source` for source provenance.

**Output:** `gold.dim_indicator` (Delta table)  
**Grain:** One row per economic indicator (7 total)  
**Surrogate key:** `indicator_key` (integer)  
**Type:** SCD Type 0 — indicator definitions are fixed  

| Column | Type | Description |
|---|---|---|
| `indicator_key` | int | Surrogate key |
| `indicator_code` | string | Natural key — column name in Silver source tables |
| `indicator_name` | string | Business-friendly display name |
| `category` | string | High-level grouping (FX, Equities, Monetary Policy, Inflation, GDP) |
| `subcategory` | string | Secondary classification |
| `unit` | string | Unit of measure |
| `aggregation_type` | string | How values are aggregated — Average or End-of-period |
| `is_higher_better` | boolean | True if a higher value is favourable — used for KPI formatting |
| `source_key` | int | FK → `gold.dim_source` |

In [ ]:
import pandas as pd

# source_key: 1=Yahoo Finance, 2=Seðlabanki Íslands, 3=Hagstofa Íslands
indicators = [
    (1, "avg_iskusd",     "ISK/USD Exchange Rate",        "FX",               "Exchange Rate", "ISK per 1 USD",   "Average",       None,  1),
    (2, "avg_iskeur",     "ISK/EUR Exchange Rate",        "FX",               "Exchange Rate", "ISK per 1 EUR",   "Average",       None,  1),
    (3, "avg_eurusd",     "EUR/USD Exchange Rate",        "FX",               "Cross Rate",    "USD per 1 EUR",   "Average",       None,  1),
    (4, "avg_omx",        "OMX Iceland All-Share Index",  "Equities",         "Stock Index",   "Index Points",    "Average",       True,  1),
    (5, "policy_rate",    "Central Bank Policy Rate",     "Monetary Policy",  "Interest Rate", "Percent (%)",     "End-of-period", None,  2),
    (6, "cpi",            "CPI Year-on-Year Inflation",   "Inflation",        "Price Index",   "Percent (%)",     "End-of-period", False, 2),
    (7, "gdp_yoy_growth", "GDP Year-on-Year Growth",      "GDP",              "Growth Rate",   "Percent (%)",     "End-of-period", True,  3),
]

df_ind = pd.DataFrame(indicators, columns=[
    "indicator_key", "indicator_code", "indicator_name",
    "category", "subcategory", "unit", "aggregation_type",
    "is_higher_better", "source_key"
])

print(df_ind.to_string(index=False))

In [ ]:
df_spark = spark.createDataFrame(df_ind)
df_spark.createOrReplaceTempView("v_dim_indicator")

if not spark.catalog.tableExists("gold.dim_indicator"):
    df_spark.write.format("delta").saveAsTable("gold.dim_indicator")
else:
    spark.sql("""
        MERGE INTO gold.dim_indicator AS target
        USING v_dim_indicator AS source
        ON target.indicator_key = source.indicator_key
        WHEN MATCHED THEN UPDATE SET
            target.indicator_code  = source.indicator_code,
            target.indicator_name  = source.indicator_name,
            target.category        = source.category,
            target.subcategory     = source.subcategory,
            target.unit            = source.unit,
            target.aggregation_type = source.aggregation_type,
            target.is_higher_better = source.is_higher_better,
            target.source_key      = source.source_key
        WHEN NOT MATCHED THEN INSERT (
            indicator_key, indicator_code, indicator_name, category, subcategory,
            unit, aggregation_type, is_higher_better, source_key
        )
        VALUES (
            source.indicator_key, source.indicator_code, source.indicator_name,
            source.category, source.subcategory, source.unit, source.aggregation_type,
            source.is_higher_better, source.source_key
        )
    """)

print(f"Saved to gold.dim_indicator - {spark.table('gold.dim_indicator').count()} rows")